# ML-05 — Feature Vector and Leakage/Privacy Check

This notebook builds a safe feature vector for the FlyRank starter dataset, documents each feature, checks leakage, records exclusions, and ends with executable self-checks. The target is `is_declining_label = (trend_direction == 'down')`.

The starter data is an export-time snapshot. Its 90-day and recent-window metrics overlap the snapshot label window, so they are suitable for export-time triage/classification only—not a claim about a future month.

## Load the starter data

Find the repository CSV from common notebook working directories and validate the required columns.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

candidate_paths = [
    Path.cwd() / 'data' / 'raw' / 'content_refresh_anonymized.csv',
    Path.cwd().parent / 'data' / 'raw' / 'content_refresh_anonymized.csv',
    Path.cwd().parent.parent / 'data' / 'raw' / 'content_refresh_anonymized.csv',
    Path.cwd() / 'content_refresh_anonymized.csv',
]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find data/raw/content_refresh_anonymized.csv')
df = pd.read_csv(data_path)
required = {'content_id', 'client_id', 'content_type', 'trend_direction'}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(f'Missing required columns: {sorted(missing_required)}')
print('Loaded:', data_path)
print('Shape:', df.shape)
print('Unique content IDs:', df['content_id'].nunique())
print('Unique clients:', df['client_id'].nunique())

## 1. Build the feature vector

The vector contains approved content, keyword, and historical activity fields. IDs, the target, target-derived fields, recent label windows, and generation metadata are excluded. Numeric missing values use medians for this demonstration; a real split must fit those medians on training data only. Structural missingness is retained with flags and categorical missingness uses `unknown`.

In [ ]:
numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
    'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d',
    'scroll_events_90d', 'days_with_impressions', 'days_with_sessions',
    'content_age_days', 'days_since_last_update', 'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]
categorical_features = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier'
]
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]
missing_flag_columns = [c for c in ['search_volume', 'competition', 'competition_level', 'cpc', 'word_count', 'char_count', 'main_intent'] if c in df.columns and df[c].isna().any()]
X = pd.DataFrame(index=df.index)
for col in numeric_features:
    values = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
    X[col] = values.fillna(values.median())
for col in missing_flag_columns:
    X[f'has_{col}'] = df[col].notna().astype('int8')
for col in categorical_features:
    X[col] = df[col].fillna('unknown').astype(str).replace({'': 'unknown', 'nan': 'unknown'})
X = pd.get_dummies(X, columns=categorical_features, dtype='int8')
print('Base numeric features:', numeric_features)
print('Categorical features:', categorical_features)
print('Missingness flags:', [c for c in X.columns if c.startswith('has_')])
print('Feature matrix shape:', X.shape)
print('Numeric matrix finite:', bool(np.isfinite(X.select_dtypes(include=np.number).to_numpy()).all()))

## 2. Feature notes (meaning, missing, categorical, available-when?)

Every base feature is documented below. The window note distinguishes fields available before export-time scoring from fields that overlap the snapshot label window. Rates are percentage-point values; `avg_position == 0` means no position data, not rank zero.

In [ ]:
descriptions = {
    'search_volume': 'Target-keyword demand estimate',
    'competition': 'Keyword competition score',
    'cpc': 'Target-keyword cost-per-click estimate',
    'word_count': 'Measured article word count',
    'char_count': 'Measured article character count',
    'impressions_90d': 'GSC impressions in trailing 90 days',
    'clicks_90d': 'GSC clicks in trailing 90 days',
    'pageviews_90d': 'GA4 pageviews in trailing 90 days',
    'sessions_90d': 'GA4 sessions in trailing 90 days',
    'users_90d': 'GA4 users in trailing 90 days',
    'engaged_sessions_90d': 'GA4 engaged sessions in trailing 90 days',
    'ai_sessions_90d': 'Sessions referred from AI tools',
    'scroll_events_90d': 'GA4 scroll events in trailing 90 days',
    'days_with_impressions': 'Days with at least one GSC impression',
    'days_with_sessions': 'Days with at least one GA4 session',
    'content_age_days': 'Days since content creation',
    'days_since_last_update': 'Days since last content update',
    'ctr': 'Clicks / impressions times 100',
    'avg_position': 'Mean GSC position; zero means no data',
    'engagement_rate': 'Engaged sessions / sessions times 100',
    'scroll_rate': 'Scroll events / pageviews times 100',
    'ai_traffic_pct': 'AI sessions / sessions times 100',
}
rows = []
for col in numeric_features:
    overlap = col in {'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'}
    rows.append({'feature': col, 'meaning': descriptions.get(col, 'Numeric content feature'), 'missing_handling': 'median; missing flag where structural', 'available_when': 'snapshot window overlaps label' if overlap else 'before export-time scoring'})
for col in categorical_features:
    rows.append({'feature': col, 'meaning': f'Categorical {col}', 'missing_handling': 'unknown category', 'available_when': 'before export-time scoring'})
notes = pd.DataFrame(rows)
print(notes.to_string(index=False))
print('All base features documented:', set(notes['feature']) == set(numeric_features + categorical_features))

## 3. The leakage hunt

The target is directly derived from `trend_direction`; `trend_pct` is also derived from the same 30-day comparison. The last/previous 30-day impression fields define the target and must not be used to claim future prediction. IDs and provider/model metadata are also excluded. The assertions below test that direct leakage is absent from the constructed matrix.

In [ ]:
label = (df['trend_direction'].astype(str).str.lower() == 'down').astype('int8')
direct_label_columns = {'trend_direction', 'trend_pct', 'is_declining_label'}
label_input_columns = {'impressions_last_30d', 'impressions_prev_30d'}
identity_columns = {'content_id', 'client_id'}
workflow_columns = {'provider_used', 'model_used'}
print('Direct label columns in X:', sorted(direct_label_columns & set(X.columns)))
print('Label-input columns in X:', sorted(label_input_columns & set(X.columns)))
print('Target prevalence:', round(float(label.mean()), 4))
assert not (direct_label_columns & set(X.columns)), 'Direct label leakage found'
assert not (identity_columns & set(X.columns)), 'Identifier leakage found'
assert not (workflow_columns & set(X.columns)), 'Workflow metadata leakage found'
leakage_audit = pd.DataFrame([
    {'item': 'target/direct derivatives', 'columns': sorted(direct_label_columns), 'status': 'PASS: excluded'},
    {'item': 'target source windows', 'columns': sorted(label_input_columns), 'status': 'WARNING: not safe for future-month prediction'},
    {'item': 'pseudonymous identifiers', 'columns': sorted(identity_columns), 'status': 'PASS: excluded'},
    {'item': 'generation metadata', 'columns': sorted(workflow_columns), 'status': 'PASS: excluded'},
])
print(leakage_audit.to_string(index=False))

## 4. What I excluded and why

These fields are excluded explicitly. IDs remain available for grouped validation only. Recent-window fields remain available for diagnostic audits, but not in this conservative feature vector because they overlap the label definition.

In [ ]:
excluded = pd.DataFrame([
    {'column': 'trend_direction', 'reason': 'Direct source of the target.'},
    {'column': 'trend_pct', 'reason': 'Derived from the target comparison windows.'},
    {'column': 'is_declining_label', 'reason': 'The target itself.'},
    {'column': 'impressions_last_30d', 'reason': 'Part of target definition and label window.'},
    {'column': 'impressions_prev_30d', 'reason': 'Part of target definition; not safe for future-month claims.'},
    {'column': 'clicks_last_30d', 'reason': 'Recent-window metric excluded from conservative vector.'},
    {'column': 'clicks_prev_30d', 'reason': 'Recent comparison-window metric excluded from conservative vector.'},
    {'column': 'sessions_last_30d', 'reason': 'Recent-window metric excluded to avoid temporal overlap.'},
    {'column': 'sessions_prev_30d', 'reason': 'Recent comparison-window metric excluded from conservative vector.'},
    {'column': 'content_id', 'reason': 'Pseudonymous identifier; context/grouping only.'},
    {'column': 'client_id', 'reason': 'Pseudonymous identifier; grouped splits only.'},
    {'column': 'provider_used', 'reason': 'Generation workflow metadata, not page quality.'},
    {'column': 'model_used', 'reason': 'Generation workflow metadata, unstable and process-revealing.'},
])
print(excluded.to_string(index=False))
print('Excluded fields found:', sorted(set(excluded['column']) & set(df.columns)))

## Self-check

- [x] Every section is filled in order.
- [x] Feature construction, fills, categories, and missingness flags are executable.
- [x] Label, identifier, and workflow leakage are tested.
- [x] Exclusions have one-line reasons.
- [x] The temporal limitation is stated honestly.
- [x] The final cell asserts the notebook's core guarantees.

In [ ]:
checks = {
    'data_loaded': len(df) > 0,
    'feature_matrix_nonempty': X.shape[1] > 0,
    'no_direct_label_leakage': not (direct_label_columns & set(X.columns)),
    'no_identifier_leakage': not (identity_columns & set(X.columns)),
    'no_workflow_metadata': not (workflow_columns & set(X.columns)),
    'all_numeric_values_finite': bool(np.isfinite(X.select_dtypes(include=np.number).to_numpy()).all()),
    'all_base_features_documented': set(notes['feature']) == set(numeric_features + categorical_features),
    'target_is_binary': set(label.unique()).issubset({0, 1}),
}
for name, passed in checks.items():
    print(f'{name}: {passed}')
assert all(checks.values()), 'One or more self-checks failed.'
print('All self-checks passed.')